In [1]:
# Imports système
import sys
import os
import time
from datetime import datetime
from typing import Dict, List, Any, Tuple
import warnings
warnings.filterwarnings('ignore')

# Ajout du projet au path
sys.path.append(os.getcwd())

# Imports scientifiques
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Configuration des graphiques
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

# Imports du projet
from src.environments import *
from src.algorithms import *
from src.utils import ExperimentRunner, HyperparameterTuner, ResultsAnalyzer

print("✅ Toutes les bibliothèques importées avec succès!")
print(f"📅 Session démarrée le: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


✅ Toutes les bibliothèques importées avec succès!
📅 Session démarrée le: 2025-07-19 18:58:10


In [3]:
# 🧮 Configuration des algorithmes avec leurs hyperparamètres optimaux
ALGORITHMS_CONFIG = {
    # Dynamic Programming
    'PolicyIteration': {
        'class': PolicyIteration,
        'category': 'Dynamic Programming',
        'params': {'gamma': 0.99, 'theta': 1e-6},
        'description': 'Itération de politique - Calcul exact avec modèle'
    },
    'ValueIteration': {
        'class': ValueIteration,
        'category': 'Dynamic Programming', 
        'params': {'gamma': 0.99, 'theta': 1e-6},
        'description': 'Itération de valeur - Calcul exact avec modèle'
    },
    
    # Monte Carlo Methods
    'OnPolicyMonteCarlo': {
        'class': OnPolicyMonteCarlo,
        'category': 'Monte Carlo',
        'params': {'episodes': 2000, 'epsilon': 0.1, 'gamma': 0.99},
        'description': 'Monte Carlo on-policy - Apprentissage par épisodes complets'
    },
    'OffPolicyMonteCarlo': {
        'class': OffPolicyMonteCarlo,
        'category': 'Monte Carlo',
        'params': {'episodes': 2000, 'epsilon': 0.1, 'gamma': 0.99, 'behavior_epsilon': 0.3},
        'description': 'Monte Carlo off-policy - Importance sampling'
    },
    
    # Temporal Difference Learning
    'QLearning': {
        'class': QLearning,
        'category': 'Temporal Difference',
        'params': {'episodes': 1500, 'alpha': 0.1, 'epsilon': 0.1, 'gamma': 0.99, 'epsilon_decay': 0.995},
        'description': 'Q-Learning - Off-policy TD optimal'
    },
    'Sarsa': {
        'class': Sarsa,
        'category': 'Temporal Difference',
        'params': {'episodes': 1500, 'alpha': 0.1, 'epsilon': 0.1, 'gamma': 0.99, 'epsilon_decay': 0.995},
        'description': 'SARSA - On-policy TD safe'
    },
    'ExpectedSarsa': {
        'class': ExpectedSarsa,
        'category': 'Temporal Difference',
        'params': {'episodes': 1500, 'alpha': 0.1, 'epsilon': 0.1, 'gamma': 0.99, 'epsilon_decay': 0.995},
        'description': 'Expected SARSA - Hybride on/off-policy'
    },
    
    # Planning Methods
    'DynaQ': {
        'class': DynaQ,
        'category': 'Planning',
        'params': {'episodes': 800, 'alpha': 0.1, 'epsilon': 0.1, 'gamma': 0.99, 'planning_steps': 10},
        'description': 'Dyna-Q - Apprentissage + Planification'
    },
    'DynaQPlus': {
        'class': DynaQPlus,
        'category': 'Planning',
        'params': {'episodes': 800, 'alpha': 0.1, 'epsilon': 0.1, 'gamma': 0.99, 'planning_steps': 10, 'kappa': 0.001},
        'description': 'Dyna-Q+ - Avec bonus d\'exploration temporel'
    }
}

# 🎮 Configuration des environnements
ENVIRONMENTS_CONFIG = {
    'LineWorld': {
        'class': LineWorld,
        'difficulty': 'Facile',
        'type': 'Navigation',
        'description': 'Monde linéaire - Navigation 1D simple'
    },
    'GridWorld': {
        'class': GridWorld,
        'difficulty': 'Moyen',
        'type': 'Navigation',
        'description': 'Grille 2D avec obstacles et objectifs'
    },
    'TwoRoundRockPaperScissors': {
        'class': TwoRoundRockPaperScissors,
        'difficulty': 'Moyen',
        'type': 'Stratégique',
        'description': 'Jeu stratégique avec adversaire adaptatif'
    },
    'MontyHallLevel1': {
        'class': MontyHallLevel1,
        'difficulty': 'Difficile',
        'type': 'Probabiliste',
        'description': 'Paradoxe de Monty Hall classique (3 portes)'
    },
    'MontyHallLevel2': {
        'class': MontyHallLevel2,
        'difficulty': 'Très Difficile',
        'type': 'Probabiliste',
        'description': 'Monty Hall étendu (5 portes, 4 décisions)'
    }
}

print(f"🧮 {len(ALGORITHMS_CONFIG)} algorithmes configurés")
print(f"🎮 {len(ENVIRONMENTS_CONFIG)} environnements configurés")
print(f"🔬 Total de combinaisons: {len(ALGORITHMS_CONFIG)} × {len(ENVIRONMENTS_CONFIG)} = {len(ALGORITHMS_CONFIG) * len(ENVIRONMENTS_CONFIG)}")

# Affichage de la configuration
print("\n📋 Algorithmes disponibles:")
for name, config in ALGORITHMS_CONFIG.items():
    print(f"  • {name:<20} [{config['category']}] - {config['description']}")

print("\n🎯 Environnements disponibles:")
for name, config in ENVIRONMENTS_CONFIG.items():
    print(f"  • {name:<30} [{config['difficulty']}] - {config['description']}")


🧮 9 algorithmes configurés
🎮 5 environnements configurés
🔬 Total de combinaisons: 9 × 5 = 45

📋 Algorithmes disponibles:
  • PolicyIteration      [Dynamic Programming] - Itération de politique - Calcul exact avec modèle
  • ValueIteration       [Dynamic Programming] - Itération de valeur - Calcul exact avec modèle
  • OnPolicyMonteCarlo   [Monte Carlo] - Monte Carlo on-policy - Apprentissage par épisodes complets
  • OffPolicyMonteCarlo  [Monte Carlo] - Monte Carlo off-policy - Importance sampling
  • QLearning            [Temporal Difference] - Q-Learning - Off-policy TD optimal
  • Sarsa                [Temporal Difference] - SARSA - On-policy TD safe
  • ExpectedSarsa        [Temporal Difference] - Expected SARSA - Hybride on/off-policy
  • DynaQ                [Planning] - Dyna-Q - Apprentissage + Planification
  • DynaQPlus            [Planning] - Dyna-Q+ - Avec bonus d'exploration temporel

🎯 Environnements disponibles:
  • LineWorld                      [Facile] - Monde linéaire

In [4]:
def test_algorithm_on_all_environments(algorithm_name: str, num_runs: int = 3):
    """
    Teste un algorithme sur tous les environnements.
    
    Args:
        algorithm_name: Nom de l'algorithme (ex: 'QLearning')
        num_runs: Nombre d'exécutions par environnement
    """
    if algorithm_name not in ALGORITHMS_CONFIG:
        print(f"❌ Algorithme '{algorithm_name}' non reconnu!")
        print(f"Algorithmes disponibles: {list(ALGORITHMS_CONFIG.keys())}")
        return None
    
    print(f"🚀 TEST DE {algorithm_name}")
    print("="*60)
    
    algo_config = ALGORITHMS_CONFIG[algorithm_name]
    results = {}
    
    # Créer le runner
    runner = ExperimentRunner(results_dir=f"experiments/{algorithm_name}")
    
    # Tester sur chaque environnement
    for env_name, env_config in ENVIRONMENTS_CONFIG.items():
        print(f"\n🎮 Test sur {env_name} [{env_config['difficulty']}]")
        
        env_rewards = []
        
        # Plusieurs runs
        for run in range(num_runs):
            try:
                result = runner.run_single_experiment(
                    algorithm_class=algo_config['class'],
                    env_class=env_config['class'],
                    algorithm_params=algo_config['params'].copy(),
                    experiment_name=f"{algorithm_name}_{env_name}_run{run+1}"
                )
                
                if 'evaluation' in result:
                    reward = result['evaluation']['mean_reward']
                    env_rewards.append(reward)
                    print(f"   Run {run+1}: {reward:.3f}")
                    
            except Exception as e:
                print(f"   Run {run+1}: ❌ Erreur - {str(e)}")
        
        # Calculer les statistiques
        if env_rewards:
            results[env_name] = {
                'mean': np.mean(env_rewards),
                'std': np.std(env_rewards),
                'best': np.max(env_rewards),
                'runs': len(env_rewards)
            }
            print(f"   📊 Moyenne: {results[env_name]['mean']:.3f} ± {results[env_name]['std']:.3f}")
        else:
            results[env_name] = {'mean': 0, 'std': 0, 'best': 0, 'runs': 0}
            print(f"   ❌ Aucun run réussi")
    
    # Afficher le résumé
    print(f"\n📈 RÉSUMÉ POUR {algorithm_name}")
    print("="*60)
    
    successful_envs = [env for env, res in results.items() if res['runs'] > 0]
    if successful_envs:
        overall_performance = np.mean([results[env]['mean'] for env in successful_envs])
        best_env = max(successful_envs, key=lambda env: results[env]['mean'])
        
        print(f"✅ Environnements maîtrisés: {len(successful_envs)}/{len(ENVIRONMENTS_CONFIG)}")
        print(f"🎯 Performance moyenne: {overall_performance:.3f}")
        print(f"🏆 Meilleur environnement: {best_env} ({results[best_env]['mean']:.3f})")
        
        # Tableau des résultats
        print(f"\n📋 Détail des performances:")
        for env_name in ENVIRONMENTS_CONFIG.keys():
            res = results[env_name]
            status = "✅" if res['runs'] > 0 else "❌"
            print(f"  {status} {env_name:<30}: {res['mean']:6.3f} ± {res['std']:.3f}")
    else:
        print("❌ Aucun environnement maîtrisé par cet algorithme")
    
    return results

print("✅ Fonction de test définie!")


✅ Fonction de test définie!


In [5]:
# 🎯 TESTEZ VOTRE ALGORITHME ICI
# Modifiez cette ligne pour changer d'algorithme:
ALGORITHM_TO_TEST = 'QLearning'

# Lancer le test (3 runs par environnement)
results = test_algorithm_on_all_environments(ALGORITHM_TO_TEST, num_runs=3)


🚀 TEST DE QLearning

🎮 Test sur LineWorld [Facile]

Expérimentation: QLearning_LineWorld_run1
Algorithme: QLearning
Environnement: LineWorld
Hyperparamètres: {'episodes': 1500, 'alpha': 0.1, 'epsilon': 0.1, 'gamma': 0.99, 'epsilon_decay': 0.995}


Q-LEARNING
Alpha: 0.1, Epsilon: 0.1


Entraînement Q-Learning: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1500/1500 [00:00<00:00, 23856.48it/s]



Entraînement terminé!
Epsilon final: 0.0100
Récompense moyenne (100 derniers): 1.00

Évaluation de la politique apprise...
   Run 1: ❌ Erreur - keys must be str, int, float, bool or None, not tuple

Expérimentation: QLearning_LineWorld_run2
Algorithme: QLearning
Environnement: LineWorld
Hyperparamètres: {'episodes': 1500, 'alpha': 0.1, 'epsilon': 0.1, 'gamma': 0.99, 'epsilon_decay': 0.995}


Q-LEARNING
Alpha: 0.1, Epsilon: 0.1


Entraînement Q-Learning: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1500/1500 [00:00<00:00, 36012.71it/s]



Entraînement terminé!
Epsilon final: 0.0100
Récompense moyenne (100 derniers): 1.00

Évaluation de la politique apprise...
   Run 2: ❌ Erreur - keys must be str, int, float, bool or None, not tuple

Expérimentation: QLearning_LineWorld_run3
Algorithme: QLearning
Environnement: LineWorld
Hyperparamètres: {'episodes': 1500, 'alpha': 0.1, 'epsilon': 0.1, 'gamma': 0.99, 'epsilon_decay': 0.995}


Q-LEARNING
Alpha: 0.1, Epsilon: 0.1


...


Entraînement terminé!
Epsilon final: 0.0100
Récompense moyenne (100 derniers): 0.77

Évaluation de la politique apprise...
   Run 1: ❌ Erreur - keys must be str, int, float, bool or None, not tuple

Expérimentation: QLearning_MontyHallLevel2_run2
Algorithme: QLearning
Environnement: MontyHallLevel2
Hyperparamètres: {'episodes': 1500, 'alpha': 0.1, 'epsilon': 0.1, 'gamma': 0.99, 'epsilon_decay': 0.995}


Q-LEARNING
Alpha: 0.1, Epsilon: 0.1


Entraînement Q-Learning: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1500/1500 [00:00<00:00, 19850.68it/s]



Entraînement terminé!
Epsilon final: 0.0100
Récompense moyenne (100 derniers): 0.77

Évaluation de la politique apprise...
   Run 2: ❌ Erreur - keys must be str, int, float, bool or None, not tuple

Expérimentation: QLearning_MontyHallLevel2_run3
Algorithme: QLearning
Environnement: MontyHallLevel2
Hyperparamètres: {'episodes': 1500, 'alpha': 0.1, 'epsilon': 0.1, 'gamma': 0.99, 'epsilon_decay': 0.995}


Q-LEARNING
Alpha: 0.1, Epsilon: 0.1


Entraînement Q-Learning: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1500/1500 [00:00<00:00, 18612.56it/s]


Entraînement terminé!
Epsilon final: 0.0100
Récompense moyenne (100 derniers): 0.76

Évaluation de la politique apprise...
   Run 3: ❌ Erreur - keys must be str, int, float, bool or None, not tuple
   ❌ Aucun run réussi

📈 RÉSUMÉ POUR QLearning
❌ Aucun environnement maîtrisé par cet algorithme


In [ ]:
# Test de SARSA
sarsa_results = test_algorithm_on_all_environments('Sarsa', num_runs=3)


In [ ]:
# Test de Dyna-Q
dynaq_results = test_algorithm_on_all_environments('DynaQ', num_runs=3)


In [ ]:
# Test de Policy Iteration
policy_results = test_algorithm_on_all_environments('PolicyIteration', num_runs=2)


In [ ]:
def compare_algorithms(results_dict):
    """
    Compare les résultats de plusieurs algorithmes.
    
    Args:
        results_dict: Dictionnaire {nom_algo: résultats}
    """
    if not results_dict:
        print("❌ Aucun résultat à comparer!")
        return
    
    print("📊 COMPARAISON DES ALGORITHMES")
    print("="*70)
    
    # Créer un tableau de comparaison
    all_envs = list(ENVIRONMENTS_CONFIG.keys())
    comparison_data = []
    
    for algo_name, results in results_dict.items():
        if results is None:
            continue
            
        row = [algo_name]
        total_score = 0
        successful_envs = 0
        
        for env in all_envs:
            if env in results and results[env]['runs'] > 0:
                score = results[env]['mean']
                row.append(f"{score:.3f}")
                total_score += score
                successful_envs += 1
            else:
                row.append("❌")
        
        # Calculer la moyenne
        avg_score = total_score / successful_envs if successful_envs > 0 else 0
        row.append(f"{avg_score:.3f}")
        row.append(f"{successful_envs}/{len(all_envs)}")
        
        comparison_data.append(row)
    
    # Afficher le tableau
    headers = ['Algorithme'] + all_envs + ['Moyenne', 'Réussis']
    
    # En-tête
    print(f"{'Algorithme':<20}", end="")
    for env in all_envs:
        print(f"{env[:8]:<10}", end="")
    print(f"{'Moyenne':<10}{'Réussis':<10}")
    print("-" * 120)
    
    # Données
    for row in comparison_data:
        print(f"{row[0]:<20}", end="")
        for i in range(1, len(all_envs) + 1):
            print(f"{row[i]:<10}", end="")
        print(f"{row[-2]:<10}{row[-1]:<10}")
    
    # Analyse
    print(f"\n🏆 ANALYSE:")
    
    # Meilleur algorithme global
    valid_algos = [(row[0], float(row[-2])) for row in comparison_data if row[-2] != "0.000"]
    if valid_algos:
        best_algo = max(valid_algos, key=lambda x: x[1])
        print(f"   Meilleur global: {best_algo[0]} (score: {best_algo[1]:.3f})")
    
    # Meilleur par environnement
    print(f"\n🎯 Meilleur par environnement:")
    for i, env in enumerate(all_envs):
        env_scores = []
        for row in comparison_data:
            if row[i+1] != "❌":
                env_scores.append((row[0], float(row[i+1])))
        
        if env_scores:
            best_for_env = max(env_scores, key=lambda x: x[1])
            print(f"   {env:<30}: {best_for_env[0]} ({best_for_env[1]:.3f})")

print("✅ Fonction de comparaison définie!")


In [ ]:
# 🔥 COMPARAISON DE VOS RÉSULTATS
# Ajoutez ici les variables de résultats que vous avez testées
results_to_compare = {
    'QLearning': results if 'results' in locals() else None,
    'Sarsa': sarsa_results if 'sarsa_results' in locals() else None,
    'DynaQ': dynaq_results if 'dynaq_results' in locals() else None,
    'PolicyIteration': policy_results if 'policy_results' in locals() else None,
}

# Nettoyer les résultats None
results_to_compare = {k: v for k, v in results_to_compare.items() if v is not None}

if len(results_to_compare) >= 2:
    compare_algorithms(results_to_compare)
else:
    print("⚠️ Vous devez tester au moins 2 algorithmes pour faire une comparaison!")
    print("   Exécutez d'abord les cellules de test ci-dessus.")
